In [ ]:
# NOTE: the original file is from https://www.kaggle.com/datasets/saurabhshahane/ecommerce-text-classification
# we're making it smaller for testing purposes

import pandas as pd

# Read the CSV file
# The file has no header, so we'll specify column names
df = pd.read_csv('ecommerceDataset.csv', header=None, names=['category', 'description'])

# Remove any rows with missing categories
df = df[df['category'].notna()]

# Group by category and get top 50 products from each category
top_products = []
for category, group in df.groupby('category'):
    # Take the first 50 products from each category
    top_50 = group.head(50)
    top_products.append(top_50)
    print(f"Category '{category}': {len(group)} total products, extracted {len(top_50)} products")

# Combine all top products
result_df = pd.concat(top_products, ignore_index=True)

# Write to new CSV file
result_df.to_csv('small_dataset.csv', index=False, header=False)

print(f"\nTotal products in small_dataset.csv: {len(result_df)}")
print(f"Categories included: {result_df['category'].unique()}")

How we want to compute user preferences per category: [tensor playground link](https://docs.vespa.ai/playground/#N4KABGBEBmkFxgNrgmUrWQPYAd5QGNIAaFDSPBdDTAF30gBUBTAOwGcsAnMLaMAIZgCAGyy0AFgEtWAczBTazALaQyqAL7qNpDNXK4GzEuogV8+mpFYMcXLABMArgVoBBEzUzGEkJR24ACgIBJVluAE9gHTBoZlCnLmZ2aIBKKgAdSFFxaTkszMgkhwKwACYAOgAGYjAsgA8RUoBmao0NNS8tDB11SzNDXyJdK0o0UzoGFgCePkFhAS5Omm7NEYh+qEGoY3XMMc2zG187RxdaACFPLx2Gf04uYNDmcK4omLiEpJSNdLQskJcUrALIAIxETmYpUqNTqkGUOFkLVaVXayx62j2m2wY2y13IBwmZnovgAquxmDw7Mw4klWARkmBAgICPZ2OxBCIRAswtwpMlUujNJi+hMcUZ8fsLESoMcoNTaWwGexJeQfFB7kEQry3tFap9aIlkmlCoDgWCIVD8ABWNq1AFiSQyJEWC2Q0q22ENJr4ACMaImqwgvT0Yu2eL2ZkJNw1DAAIuIwKdnK5ePwDUb2BUwBdmCEnBSwPECBIFEplLxWCIImBQcwxHIObQsJXmDyXpFagIuWBJG3cpT269+RyAO5YJwiBzCLCsWhcKSgpxKMCjxSlqoVIXBkWhrzi3y7MXRm7WBgOcQABXsKfcqu8DHYTmUgSfL+T5zcYAAVEmkor6WSWptQ7N5Un1eJDW+QVA13VBsXDI99xPfc5UgC9aA-VwrkjB9fDfV9n0CLDLh-P8aUpJUgKHSJwNiSDMxgrptBQABdEANCAA)

In [ ]:
import json
import uuid
import pandas as pd

# Convert small_dataset.csv to Vespa feed format (JSONL)
def convert_to_vespa_feed(csv_path, jsonl_path):
    """
    Convert small_dataset.csv to JSONL format for Vespa ingestion.
    Similar to groceries.jsonl format.
    """
    # Read CSV using pandas (handles quoted fields properly)
    df = pd.read_csv(csv_path, header=None, names=['category', 'description'])
    
    with open(jsonl_path, 'w', encoding='utf-8') as outfile:
        for _, row in df.iterrows():
            category = str(row['category']).strip() if pd.notna(row['category']) else None
            description = str(row['description']).strip() if pd.notna(row['description']) else None
            
            # Skip rows with missing category or description
            if not category or not description or category == 'nan' or description == 'nan':
                continue
            
            # Generate UUID for this product
            product_id = str(uuid.uuid4())
            
            # Create the JSON object in Vespa feed format
            json_obj = {
                "put": f"id:ecommerce:product::{product_id}",
                "fields": {
                    "category": category,
                    "description": description
                }
            }
            
            # Write as a single line to the JSONL file
            outfile.write(json.dumps(json_obj) + '\n')

# Convert the file
csv_path = 'small_dataset.csv'
jsonl_path = 'small_dataset.jsonl'

print(f"Converting {csv_path} to {jsonl_path}...")
convert_to_vespa_feed(csv_path, jsonl_path)

# Count lines to verify
with open(jsonl_path, 'r', encoding='utf-8') as f:
    line_count = sum(1 for _ in f)

print(f"Conversion complete! Output written to {jsonl_path}")
print(f"Total products in {jsonl_path}: {line_count}")


In [ ]:
# Load environment variables and initialize OpenAI client
import os
import json
from pathlib import Path
from openai import OpenAI
import time

# Load environment variables
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    print("Note: python-dotenv not installed. Install with 'pip install python-dotenv' to use .env files.")

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
if not OPENAI_API_KEY:
    print("Warning: OPENAI_API_KEY not found in environment variables.")
    print("Please add your OpenAI API key to the .env file before running the feature extraction cells.")
    print("Example .env file content:")
    print("OPENAI_API_KEY=your-api-key-here")
else:
    # Initialize OpenAI client
    client = OpenAI(api_key=OPENAI_API_KEY)
    print("OpenAI client initialized successfully.")

In [ ]:
def extract_features_with_openai(category, description, max_retries=3):
    """
    Extract relevant features/themes from a product description using OpenAI.
    Returns a list of feature objects with address and value.
    """
    prompt = f"""
Analyze this product description and identify the most relevant features/themes that describe what this product is about.

Category: {category}
Description: {description}

Please identify 3-8 key features/themes that best characterize this product. For each feature, provide:
- The feature name (e.g., "memoir", "fantasy", "self-help", "cooking", "technology", etc.)
- A relevance score from 1-10, where:
  * 10 = This product is TOTALLY about this feature
  * 7-9 = This is a primary/strong feature of the product
  * 4-6 = This is a secondary/moderate feature
  * 1-3 = There's a vague connection but not central

Only include features with scores of 1 or higher. Do not include scores of 0.

Respond with a JSON array of objects, each with "feature" and "score" fields.

Example response format:
[
  {{"feature": "memoir", "score": 9}},
  {{"feature": "self-improvement", "score": 7}},
  {{"feature": "spirituality", "score": 8}}
]
"""

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model="gpt-5-mini",
                input=prompt
            )

            response_text = response.output_text.strip()

            # Try to parse JSON
            try:
                features = json.loads(response_text)
                if not isinstance(features, list):
                    raise ValueError("Response is not a JSON array")

                # Convert to the required format
                formatted_features = []
                for item in features:
                    if isinstance(item, dict) and 'feature' in item and 'score' in item:
                        score = int(item['score'])
                        if 1 <= score <= 10:  # Only include valid scores
                            formatted_features.append({
                                "address": {
                                    "category": category,
                                    "features": item['feature']
                                },
                                "value": score
                            })

                return formatted_features

            except (json.JSONDecodeError, ValueError, KeyError) as e:
                print(f"Failed to parse OpenAI response (attempt {attempt + 1}): {e}")
                print(f"Response: {response_text}")
                if attempt == max_retries - 1:
                    return []  # Return empty list if all attempts fail

        except Exception as e:
            print(f"OpenAI API error (attempt {attempt + 1}): {e}")
            if attempt < max_retries - 1:
                time.sleep(1)  # Wait before retry
            else:
                return []

    return []

In [ ]:
# Read the small_dataset.jsonl file and add features using OpenAI
input_file = 'small_dataset.jsonl'
output_file = 'small_dataset_with_features.jsonl'

print(f"Reading {input_file} and adding features using OpenAI...")
print("This may take a while as we process 200 documents with API calls...")

processed_count = 0
total_documents = 0

# Count total documents first
with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:
        total_documents += 1

print(f"Found {total_documents} documents to process")

# Process documents and add features
with open(input_file, 'r', encoding='utf-8') as infile, \
     open(output_file, 'w', encoding='utf-8') as outfile:

    for line_num, line in enumerate(infile, 1):
        try:
            # Parse the JSON line
            doc = json.loads(line.strip())

            # Extract category and description
            category = doc['fields']['category']
            description = doc['fields']['description']

            # Get features from OpenAI
            features = extract_features_with_openai(category, description)

            # Add features to the document
            doc['fields']['features'] = features

            # Write the updated document
            outfile.write(json.dumps(doc, ensure_ascii=False) + '\n')

            processed_count += 1

            # Progress update every 10 documents
            if processed_count % 10 == 0:
                print(f"\rProcessed {processed_count}/{total_documents} documents...", end='', flush=True)

        except Exception as e:
            print(f"\nError processing document {line_num}: {e}")
            # Write original document if processing fails
            try:
                outfile.write(line)
                processed_count += 1
            except:
                pass

print(f"\n\nProcessing complete!")
print(f"Processed {processed_count} documents")
print(f"Output written to {output_file}")

# Show a sample of the results
print("\nSample of processed documents:")
with open(output_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:  # Show first 3 examples
            break
        doc = json.loads(line.strip())
        print(f"\nDocument {i+1}:")
        print(f"  Category: {doc['fields']['category']}")
        print(f"  Features: {doc['fields']['features']}")
        print(f"  Description preview: {doc['fields']['description'][:100]}...")